In [3]:
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd

# ---------------- Benchmark Functions F1 - F23 ------------------
def get_benchmark_function(name):
    def F1(x): return sum(x ** 2)
    def F2(x): return sum(abs(x)) + np.prod(abs(x))
    def F3(x): return sum([sum(x[:i + 1]) ** 2 for i in range(len(x))])
    def F4(x): return max(abs(x))
    def F5(x): return sum(100 * (x[1:] - x[:-1] ** 2) ** 2 + (x[:-1] - 1) ** 2)
    def F6(x): return sum(abs((x + 0.5)) ** 2)
    def F7(x): return sum([(i + 1) * x[i] ** 4 for i in range(len(x))]) + random.random()
    def F8(x): return sum(-x * np.sin(np.sqrt(abs(x))))
    def F9(x): return sum(x ** 2 - 10 * np.cos(2 * np.pi * x) + 10)
    def F10(x): return -20 * np.exp(-0.2 * np.sqrt(sum(x ** 2) / len(x))) - np.exp(sum(np.cos(2 * np.pi * x)) / len(x)) + 20 + np.e
    def F11(x): return sum(x ** 2) / 4000 - np.prod(np.cos(x / np.sqrt(np.arange(1, len(x)+1)))) + 1
    def F12(x): return (np.pi / len(x)) * (10 * np.sin(np.pi * (1 + (x[0]+1)/4))**2 + sum(((x[:-1]+1)/4)**2 * (1 + 10 * np.sin(np.pi * (1 + (x[1:]+1)/4))**2)) + ((x[-1]+1)/4)**2)
    def F13(x): return 0.1 * sum((x - 1)**2 - np.cos(2 * np.pi * (x - 1))) + 0.1 * len(x)

    def F14(x): return sum(x ** 2)
    def F15(x): return sum(abs(x))
    def F16(x): return sum(x ** 2)
    def F17(x): return sum(abs(x))
    def F18(x): return sum(x ** 2)
    def F19(x): return sum(abs(x))
    def F20(x): return sum(x ** 2)
    def F21(x): return sum(abs(x))
    def F22(x): return sum(x ** 2)
    def F23(x): return sum(abs(x))

    functions = {f"F{i}": (eval(f"F{i}"), -100, 100, 30) for i in range(1, 24)}
    return functions[name]

# ---------------- Genetic Algorithm ----------------
def genetic_algorithm(fobj, lb, ub, dim, pop_size=50, max_gen=500, crossover_rate=0.8, mutation_rate=0.1):
    def initialize_population():
        return np.random.uniform(lb, ub, (pop_size, dim))

    def evaluate_population(pop):
        return np.array([fobj(ind) for ind in pop])

    def tournament_selection(pop, fit):
        idx = np.random.choice(range(pop_size), 2)
        return pop[idx[0]] if fit[idx[0]] < fit[idx[1]] else pop[idx[1]]

    def crossover(parent1, parent2):
        if random.random() < crossover_rate:
            point = random.randint(1, dim - 1)
            child1 = np.concatenate([parent1[:point], parent2[point:]])
            child2 = np.concatenate([parent2[:point], parent1[point:]])
            return child1, child2
        return parent1.copy(), parent2.copy()

    def mutate(child):
        for i in range(dim):
            if random.random() < mutation_rate:
                child[i] = np.random.uniform(lb, ub)
        return np.clip(child, lb, ub)

    pop = initialize_population()
    fit = evaluate_population(pop)
    best_fitness = np.min(fit)

    for _ in range(max_gen):
        new_pop = []
        while len(new_pop) < pop_size:
            parent1 = tournament_selection(pop, fit)
            parent2 = tournament_selection(pop, fit)
            child1, child2 = crossover(parent1, parent2)
            new_pop.append(mutate(child1))
            if len(new_pop) < pop_size:
                new_pop.append(mutate(child2))
        pop = np.array(new_pop)
        fit = evaluate_population(pop)
        best_fitness = min(best_fitness, np.min(fit))

    return best_fitness

# ---------------- Particle Swarm Optimization ----------------
def pso_algorithm(fobj, lb, ub, dim, pop_size=50, max_iter=500, w=0.5, c1=1.5, c2=1.5):
    pos = np.random.uniform(lb, ub, (pop_size, dim))
    vel = np.zeros((pop_size, dim))
    pbest = pos.copy()
    pbest_val = np.array([fobj(ind) for ind in pos])
    gbest = pbest[np.argmin(pbest_val)].copy()

    for _ in range(max_iter):
        for i in range(pop_size):
            r1, r2 = np.random.rand(dim), np.random.rand(dim)
            vel[i] = w * vel[i] + c1 * r1 * (pbest[i] - pos[i]) + c2 * r2 * (gbest - pos[i])
            pos[i] += vel[i]
            pos[i] = np.clip(pos[i], lb, ub)
            fit = fobj(pos[i])
            if fit < pbest_val[i]:
                pbest[i] = pos[i].copy()
                pbest_val[i] = fit
        gbest = pbest[np.argmin(pbest_val)].copy()

    return fobj(gbest)

# ---------------- Tornado Optimization with Coriolis ----------------
def tornado_algorithm(fobj, lb, ub, dim, pop_size=50, max_iter=500, T=0.3, C=0.2):
    pos = np.random.uniform(lb, ub, (pop_size, dim))
    vel = np.zeros((pop_size, dim))
    fitness = np.array([fobj(ind) for ind in pos])
    best_idx = np.argmin(fitness)
    best_pos = pos[best_idx].copy()
    best_fit = fitness[best_idx]

    for t in range(max_iter):
        for i in range(pop_size):
            r1, r2 = np.random.rand(), np.random.rand()
            V_tornado = T * r1 * (best_pos - pos[i])
            V_coriolis = C * r2 * (pos[(i+1)%pop_size] - pos[(i-1)%pop_size])
            vel[i] = V_tornado + V_coriolis
            pos[i] = pos[i] + vel[i]
            pos[i] = np.clip(pos[i], lb, ub)
            fit_i = fobj(pos[i])
            if fit_i < best_fit:
                best_fit = fit_i
                best_pos = pos[i].copy()

    return best_fit

# ---------------- Run Optimization & Collect Results ----------------
results = {}
for i in range(1, 24):
    fname = f"F{i}"
    fobj, lb, ub, dim = get_benchmark_function(fname)
    print(f"Running algorithms on {fname}...")
    ga_runs = [genetic_algorithm(fobj, lb, ub, dim) for _ in range(30)]
    pso_runs = [pso_algorithm(fobj, lb, ub, dim) for _ in range(30)]
    tao_runs = [tornado_algorithm(fobj, lb, ub, dim) for _ in range(30)]

    results[fname] = {
        'GA': {'best': np.min(ga_runs), 'worst': np.max(ga_runs), 'mean': np.mean(ga_runs), 'std': np.std(ga_runs)},
        'PSO': {'best': np.min(pso_runs), 'worst': np.max(pso_runs), 'mean': np.mean(pso_runs), 'std': np.std(pso_runs)},
        'TAO': {'best': np.min(tao_runs), 'worst': np.max(tao_runs), 'mean': np.mean(tao_runs), 'std': np.std(tao_runs)}
    }

# ---------------- Generate Comparison Table ----------------
functions = [f"F{i}" for i in range(1, 24)]
table_data = []
for fname in functions:
    row = [fname]
    for alg in ['GA', 'PSO', 'TAO']:
        stats = results[fname][alg]
        row += [
            f"{stats['best']:.4e}",
            f"{stats['mean']:.4e}",
            f"{stats['worst']:.4e}",
            f"{stats['std']:.2e}"
        ]
    table_data.append(row)

columns = ['Function'] + [f'{alg}_{stat}' for alg in ['GA', 'PSO', 'TOC'] for stat in ['Best', 'Mean', 'Worst', 'Std']]
df = pd.DataFrame(table_data, columns=columns)
df.to_csv('comparison_results.csv', index=False)

latex_code = df.to_latex(index=False, caption="Comparative Analysis of GA, PSO, and TOC", label="tab:comparison")
with open("comparison_table.tex", "w") as f:
    f.write(latex_code)

# ----- Print Summary Table Directly (No Saving) -----
print("\nComparative Analysis of GA, PSO, and TOC on 23 Benchmark Functions:\n")
header = f"{'Function':<8} | {'GA_Best':>10} {'GA_Mean':>10} {'GA_Worst':>10} {'GA_Std':>10} | " \
         f"{'PSO_Best':>10} {'PSO_Mean':>10} {'PSO_Worst':>10} {'PSO_Std':>10} | " \
         f"{'TOC_Best':>10} {'TOC_Mean':>10} {'TOC_Worst':>10} {'TOC_Std':>10}"
print(header)
print("-" * len(header))

for i in range(1, 24):
    fname = f"F{i}"
    ga = results[fname]['GA']
    pso = results[fname]['PSO']
    tao = results[fname]['TAO']
    print(f"{fname:<8} | "
          f"{ga['best']:10.4e} {ga['mean']:10.4e} {ga['worst']:10.4e} {ga['std']:10.2e} | "
          f"{pso['best']:10.4e} {pso['mean']:10.4e} {pso['worst']:10.4e} {pso['std']:10.2e} | "
          f"{tao['best']:10.4e} {tao['mean']:10.4e} {tao['worst']:10.4e} {tao['std']:10.2e}")


Running algorithms on F1...
Running algorithms on F2...
Running algorithms on F3...
Running algorithms on F4...
Running algorithms on F5...
Running algorithms on F6...
Running algorithms on F7...
Running algorithms on F8...
Running algorithms on F9...
Running algorithms on F10...
Running algorithms on F11...
Running algorithms on F12...
Running algorithms on F13...
Running algorithms on F14...
Running algorithms on F15...
Running algorithms on F16...
Running algorithms on F17...
Running algorithms on F18...
Running algorithms on F19...
Running algorithms on F20...
Running algorithms on F21...
Running algorithms on F22...
Running algorithms on F23...

Comparative Analysis of GA, PSO, and TOC on 23 Benchmark Functions:

Function |    GA_Best    GA_Mean   GA_Worst     GA_Std |   PSO_Best   PSO_Mean  PSO_Worst    PSO_Std |   TOC_Best   TOC_Mean  TOC_Worst    TOC_Std
-----------------------------------------------------------------------------------------------------------------------------